In [1]:
from pathlib import Path

import numpy as np

from nicht_riemann_data.transforms import spacings


DATA_FILE = Path("data/raw/zeros1")

assert DATA_FILE.exists(), f"Missing data file: {DATA_FILE}"

gamma = np.loadtxt(DATA_FILE, dtype=np.float64)

assert gamma.ndim == 1
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

delta = spacings(gamma)

assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros:", len(gamma))
print("spacings:", len(delta))
print("OK")

zeros: 100000
spacings: 99999
OK


In [2]:
BLOCK_SIZE = 1000

num_blocks = len(delta) // BLOCK_SIZE

local_mean = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    local_mean[start:end] = np.mean(delta[start:end])

remainder_start = num_blocks * BLOCK_SIZE

if remainder_start < len(delta):
    local_mean[remainder_start:] = np.mean(delta[remainder_start:])

unfolded = delta / local_mean

assert unfolded.shape == delta.shape
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)

print("unfolded:", len(unfolded))
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))

unfolded: 99999
mean: 0.9999999999999997
std : 0.40186713998930124


In [3]:
SEED = 20260831

rng = np.random.default_rng(SEED)

print("seed:", SEED)
print("OK")

seed: 20260831
OK


In [4]:
surrogate_shuffle = unfolded.copy()

rng.shuffle(surrogate_shuffle)

assert surrogate_shuffle.shape == unfolded.shape
assert np.array_equal(
    np.sort(surrogate_shuffle),
    np.sort(unfolded),
)

print("shuffle surrogate created")
print("same values:", np.array_equal(
    np.sort(surrogate_shuffle),
    np.sort(unfolded),
))

shuffle surrogate created
same values: True


In [5]:
surrogate_iid = rng.choice(
    unfolded,
    size=len(unfolded),
    replace=True,
)

assert surrogate_iid.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_iid))
assert np.all(surrogate_iid > 0)

print("iid surrogate created")
print("N:", len(surrogate_iid))

iid surrogate created
N: 99999


In [6]:
surrogate_uniform = rng.uniform(
    0.0,
    2.0,
    size=len(unfolded),
)

assert surrogate_uniform.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_uniform))
assert np.all(surrogate_uniform >= 0)

print("uniform surrogate created")
print("N:", len(surrogate_uniform))
print("mean:", np.mean(surrogate_uniform))

uniform surrogate created
N: 99999
mean: 0.9963498059533168


In [7]:
datasets = {
    "observed": unfolded,
    "shuffle": surrogate_shuffle,
    "iid": surrogate_iid,
    "uniform": surrogate_uniform,
}

for name, values in datasets.items():
    print(
        f"{name:>8} : "
        f"mean={np.mean(values):.6f}  "
        f"std={np.std(values):.6f}  "
        f"min={np.min(values):.6f}  "
        f"max={np.max(values):.6f}"
    )

observed : mean=1.000000  std=0.401867  min=0.021865  max=4.897535
 shuffle : mean=1.000000  std=0.401867  min=0.021865  max=4.897535
     iid : mean=0.999474  std=0.403891  min=0.028739  max=4.897535
 uniform : mean=0.996350  std=0.578363  min=0.000022  max=1.999996


In [8]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]

for name, values in datasets.items():
    percentiles = np.percentile(values, percentile_levels)

    print(f"\n{name}")

    for p, value in zip(percentile_levels, percentiles):
        print(f"{p:>3}% : {value:.12f}")


observed
  0% : 0.021865470791
  1% : 0.234538581554
  5% : 0.398679540794
 25% : 0.710630119302
 50% : 0.965046539834
 75% : 1.252004821079
 95% : 1.720129557467
 99% : 2.072834102416
100% : 4.897535110823

shuffle
  0% : 0.021865470791
  1% : 0.234538581554
  5% : 0.398679540794
 25% : 0.710630119302
 50% : 0.965046539834
 75% : 1.252004821079
 95% : 1.720129557467
 99% : 2.072834102416
100% : 4.897535110823

iid
  0% : 0.028738958407
  1% : 0.233440926249
  5% : 0.394733943651
 25% : 0.707907998415
 50% : 0.963956848072
 75% : 1.252572194564
 95% : 1.724607637273
 99% : 2.080705624810
100% : 4.897535110823

uniform
  0% : 0.000022093503
  1% : 0.019529223307
  5% : 0.098284140741
 25% : 0.493116824048
 50% : 0.994745096944
 75% : 1.497543650447
 95% : 1.899403058055
 99% : 1.979250994675
100% : 1.999995756364


In [9]:
assert np.array_equal(
    np.sort(surrogate_shuffle),
    np.sort(unfolded),
)

assert np.isclose(
    np.mean(surrogate_shuffle),
    np.mean(unfolded),
)

assert np.isclose(
    np.std(surrogate_shuffle),
    np.std(unfolded),
)

print("shuffle distribution invariant: OK")

shuffle distribution invariant: OK


In [10]:
def lag1_correlation(values):
    x = values[:-1]
    y = values[1:]

    return np.corrcoef(x, y)[0, 1]


for name, values in datasets.items():
    print(
        f"{name:>8} : "
        f"lag-1 correlation = {lag1_correlation(values):.8f}"
    )

observed : lag-1 correlation = -0.35184843
 shuffle : lag-1 correlation = -0.00157181
     iid : lag-1 correlation = 0.00467110
 uniform : lag-1 correlation = -0.00266665


In [11]:
for name, values in datasets.items():
    differences = np.diff(values)

    print(
        f"{name:>8} : "
        f"diff std={np.std(differences):.8f}  "
        f"diff mean={np.mean(differences):.8f}"
    )

observed : diff std=0.66063465  diff mean=-0.00004050
 shuffle : diff std=0.56876294  diff mean=-0.00001520
     iid : diff std=0.56985504  diff mean=-0.00000193
 uniform : diff std=0.81902021  diff mean=0.00000556


In [12]:
block_means = {}

for name, values in datasets.items():
    means = np.array([
        np.mean(values[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
        for i in range(len(values) // BLOCK_SIZE)
    ])

    block_means[name] = means

    print(
        f"{name:>8} : "
        f"block mean min={means.min():.8f}  "
        f"max={means.max():.8f}  "
        f"std={means.std():.8f}"
    )

observed : block mean min=1.00000000  max=1.00000000  std=0.00000000
 shuffle : block mean min=0.96724488  max=1.03283064  std=0.01146667
     iid : block mean min=0.97380120  max=1.03001157  std=0.01122801
 uniform : block mean min=0.95659857  max=1.04522138  std=0.01716424


In [13]:
rng_check = np.random.default_rng(SEED)

check_shuffle = unfolded.copy()
rng_check.shuffle(check_shuffle)

assert np.array_equal(
    check_shuffle,
    surrogate_shuffle,
)

print("randomized surrogate is reproducible: OK")

randomized surrogate is reproducible: OK


In [14]:
print("=== 04_surrogates summary ===")

print("observed N:", len(unfolded))
print("seed:", SEED)

print("\nlag-1 correlation:")

for name, values in datasets.items():
    print(
        f"  {name:>8}: "
        f"{lag1_correlation(values): .8f}"
    )

print("\nblock-mean std:")

for name, means in block_means.items():
    print(
        f"  {name:>8}: "
        f"{np.std(means): .8f}"
    )

print("\nALL BASIC INVARIANTS PASSED")

=== 04_surrogates summary ===
observed N: 99999
seed: 20260831

lag-1 correlation:
  observed: -0.35184843
   shuffle: -0.00157181
       iid:  0.00467110
   uniform: -0.00266665

block-mean std:
  observed:  0.00000000
   shuffle:  0.01146667
       iid:  0.01122801
   uniform:  0.01716424

ALL BASIC INVARIANTS PASSED
